Fetch and store raw weather data

In [0]:
import requests
import json

cities = [
    {"name": "sao_paulo", "state": "SP", "lat": -23.5719, "lon": -46.6334},
    {"name": "rio_de_janeiro", "state": "RJ", "lat": -22.9228, "lon": -43.3238},
    {"name": "belo_horizonte", "state": "MG", "lat": -19.9090, "lon": -43.9567},
    {"name": "brasilia", "state": "DF", "lat": -15.8112, "lon": -47.9713},
    {"name": "curitiba", "state": "PR", "lat": -25.4533, "lon": -49.2742}
]

raw_path = "/Volumes/workspace/default/logisense_raw/weather/"

dbutils.fs.mkdirs(raw_path)

url = "https://archive-api.open-meteo.com/v1/archive"

for city in cities:
    params = {
        "latitude": city["lat"],
        "longitude": city["lon"],
        "start_date": "2016-09-01",
        "end_date": "2018-10-31",
        "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
        "timezone": "auto"
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    weather_data = response.json()

    file_path = raw_path + city["name"] + ".json"

    dbutils.fs.put(
        file_path,
        json.dumps(weather_data),
        overwrite=True
    )

    print(
        city["name"],
        "| Records:", len(weather_data["daily"]["time"]),
        "| Saved successfully"
    )

print("\nRAW WEATHER INGESTION COMPLETED!")

Wrote 25018 bytes.
sao_paulo | Records: 791 | Saved successfully
Wrote 24847 bytes.
rio_de_janeiro | Records: 791 | Saved successfully
Wrote 24962 bytes.
belo_horizonte | Records: 791 | Saved successfully
Wrote 24961 bytes.
brasilia | Records: 791 | Saved successfully
Wrote 25012 bytes.
curitiba | Records: 791 | Saved successfully

RAW WEATHER INGESTION COMPLETED!


Create the Weather Bronze table

In [0]:
from pyspark.sql import functions as F

weather_path = "/Volumes/workspace/default/logisense_raw/weather/"

weather = spark.read.option("multiline", True).json(weather_path)

weather.printSchema()


root
 |-- daily: struct (nullable = true)
 |    |-- precipitation_sum: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m_mean: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- wind_speed_10m_max: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- daily_units: struct (nullable = true)
 |    |-- precipitation_sum: string (nullable = true)
 |    |-- temperature_2m_mean: string (nullable = true)
 |    |-- time: string (nullable = true)
 |    |-- wind_speed_10m_max: string (nullable = true)
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = true)
 |-- utc_offset_seconds: long (

Bronze Weather table

In [0]:
from pyspark.sql import functions as F

weather = (
    spark.read
    .option("multiline", True)
    .json(weather_path)
    .select("*", F.col("_metadata.file_path").alias("_source_file"))
)

weather_bronze = (
    weather
    .withColumn(
        "city",
        F.regexp_extract(F.col("_source_file"), r"([^/]+)\.json$", 1)
    )
    .withColumn(
        "daily_record",
        F.explode(
            F.arrays_zip(
                F.col("daily.time"),
                F.col("daily.temperature_2m_mean"),
                F.col("daily.precipitation_sum"),
                F.col("daily.wind_speed_10m_max")
            )
        )
    )
    .select(
        "city",
        F.to_date(F.col("daily_record.time")).alias("weather_date"),
        F.col("daily_record.temperature_2m_mean").alias("temperature_c"),
        F.col("daily_record.precipitation_sum").alias("precipitation_mm"),
        F.col("daily_record.wind_speed_10m_max").alias("max_wind_speed_kmh"),
        "latitude",
        "longitude",
        "elevation"
    )
)

weather_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.bronze_weather"
)

print("Bronze Weather table created successfully!")
print("Total records:", weather_bronze.count())
print("Distinct cities:", weather_bronze.select("city").distinct().count())

display(weather_bronze.orderBy("city", "weather_date").limit(10))

Bronze Weather table created successfully!
Total records: 3955
Distinct cities: 5


city,weather_date,temperature_c,precipitation_mm,max_wind_speed_kmh,latitude,longitude,elevation
belo_horizonte,2016-09-01,21.8,5.9,9.9,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-02,20.7,0.1,15.8,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-03,20.2,0.0,18.7,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-04,20.5,0.0,14.7,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-05,20.5,0.0,12.7,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-06,21.4,0.0,11.1,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-07,22.8,0.0,10.7,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-08,23.5,0.0,9.4,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-09,22.3,2.3,12.1,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-10,21.1,0.9,18.2,-19.9297,-43.966034,862.0


In [0]:
from pyspark.sql import functions as F

weather_bronze = spark.table("workspace.default.bronze_weather")

weather_bronze.select(
    F.count("*").alias("total_records"),
    F.countDistinct("city").alias("unique_cities"),
    F.countDistinct("weather_date").alias("unique_dates"),
    F.sum(F.when(F.col("weather_date").isNull(), 1).otherwise(0)).alias("missing_dates"),
    F.sum(F.when(F.col("temperature_c").isNull(), 1).otherwise(0)).alias("missing_temperature"),
    F.sum(F.when(F.col("precipitation_mm").isNull(), 1).otherwise(0)).alias("missing_precipitation"),
    F.sum(F.when(F.col("max_wind_speed_kmh").isNull(), 1).otherwise(0)).alias("missing_wind")
).display()

display(
    weather_bronze.groupBy("city")
    .agg(
        F.count("*").alias("records"),
        F.min("weather_date").alias("start_date"),
        F.max("weather_date").alias("end_date")
    )
    .orderBy("city")
)

total_records,unique_cities,unique_dates,missing_dates,missing_temperature,missing_precipitation,missing_wind
3955,5,791,0,0,0,0


city,records,start_date,end_date
belo_horizonte,791,2016-09-01,2018-10-31
brasilia,791,2016-09-01,2018-10-31
curitiba,791,2016-09-01,2018-10-31
rio_de_janeiro,791,2016-09-01,2018-10-31
sao_paulo,791,2016-09-01,2018-10-31


Silver Weather

In [0]:
from pyspark.sql import functions as F

weather_bronze = spark.table("workspace.default.bronze_weather")

weather_silver = (
    weather_bronze
    .select(
        F.lower(F.trim(F.col("city"))).alias("city"),
        F.to_date("weather_date").alias("weather_date"),
        F.col("temperature_c").cast("double"),
        F.col("precipitation_mm").cast("double"),
        F.col("max_wind_speed_kmh").cast("double"),
        F.col("latitude").cast("double"),
        F.col("longitude").cast("double"),
        F.col("elevation").cast("double")
    )
    .filter(
        F.col("city").isNotNull() &
        F.col("weather_date").isNotNull() &
        F.col("temperature_c").isNotNull() &
        F.col("precipitation_mm").isNotNull() &
        F.col("max_wind_speed_kmh").isNotNull()
    )
    .dropDuplicates(["city", "weather_date"])
)

weather_silver.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.silver_weather"
)

print("Silver Weather table created!")
print("Total records:", weather_silver.count())
print("Unique cities:", weather_silver.select("city").distinct().count())

display(weather_silver.orderBy("city", "weather_date").limit(10))

Silver Weather table created!
Total records: 3955
Unique cities: 5


city,weather_date,temperature_c,precipitation_mm,max_wind_speed_kmh,latitude,longitude,elevation
belo_horizonte,2016-09-01,21.8,5.9,9.9,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-02,20.7,0.1,15.8,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-03,20.2,0.0,18.7,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-04,20.5,0.0,14.7,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-05,20.5,0.0,12.7,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-06,21.4,0.0,11.1,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-07,22.8,0.0,10.7,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-08,23.5,0.0,9.4,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-09,22.3,2.3,12.1,-19.9297,-43.966034,862.0
belo_horizonte,2016-09-10,21.1,0.9,18.2,-19.9297,-43.966034,862.0


In [0]:
orders = spark.table("workspace.default.silver_orders")
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_duration_days: double (nullable = true)
 |-- delivery_delay_days: double (nullable = true)
 |-- delivery_performance: string (nullable = true)
 |-- extreme_delay_flag: boolean (nullable = true)



weather-related Gold analysis

In [0]:
from pyspark.sql import functions as F

weather = spark.table("workspace.default.silver_weather")
orders = spark.table("workspace.default.silver_orders")
customers = spark.table("workspace.default.silver_customers")

weather_monthly = (
    weather
    .withColumn("city_key", F.regexp_replace("city", "_", " "))
    .withColumn("month", F.date_format("weather_date", "yyyy-MM"))
    .groupBy("city_key", "month")
    .agg(
        F.round(F.avg("temperature_c"), 2).alias("avg_temperature_c"),
        F.round(F.sum("precipitation_mm"), 2).alias("total_rainfall_mm"),
        F.round(F.avg("max_wind_speed_kmh"), 2).alias("avg_wind_speed_kmh")
    )
)

delivery_monthly = (
    orders
    .filter(F.col("order_status") == "delivered")
    .join(
        customers.select("customer_id", "customer_city"),
        "customer_id",
        "inner"
    )
    .withColumn("city_key", F.lower(F.trim(F.col("customer_city"))))
    .withColumn("month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .groupBy("city_key", "month")
    .agg(
        F.countDistinct("order_id").alias("delivered_orders"),
        F.sum(
            F.when(F.col("delivery_performance") == "Late", 1).otherwise(0)
        ).alias("late_orders"),
        F.round(F.avg("delivery_duration_days"), 2).alias("avg_delivery_days")
    )
    .withColumn(
        "late_delivery_rate",
        F.round(F.col("late_orders") / F.col("delivered_orders") * 100, 2)
    )
)

gold_weather_delivery = (
    delivery_monthly
    .join(weather_monthly, ["city_key", "month"], "inner")
)

gold_weather_delivery.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_weather_delivery_analysis"
)

print("Gold Weather Delivery Analysis created!")
print("Total records:", gold_weather_delivery.count())

display(
    gold_weather_delivery
    .orderBy(F.desc("delivered_orders"))
    .limit(15)
)

Gold Weather Delivery Analysis created!
Total records: 106


city_key,month,delivered_orders,late_orders,avg_delivery_days,late_delivery_rate,avg_temperature_c,total_rainfall_mm,avg_wind_speed_kmh
sao paulo,2018-08,1269,229,5.75,18.05,15.76,57.3,19.07
sao paulo,2018-05,1191,66,7.55,5.54,17.93,21.0,20.19
sao paulo,2018-04,1135,28,6.74,2.47,19.77,28.3,19.55
sao paulo,2018-03,1127,119,9.32,10.56,22.34,118.2,18.18
sao paulo,2017-11,1081,95,10.19,8.79,19.62,104.3,22.66
sao paulo,2018-01,1066,44,9.11,4.13,21.74,227.0,19.57
sao paulo,2018-07,1059,70,6.47,6.61,17.13,18.7,17.21
sao paulo,2018-06,1040,10,6.11,0.96,17.35,54.0,17.69
sao paulo,2018-02,1004,79,10.43,7.87,21.02,64.9,21.54
sao paulo,2017-12,820,27,9.02,3.29,21.36,171.8,21.21
